# Bronze Orchestrator: master_pdv
Orchestrates ingestion, validation, and monitoring for the Bronze layer of master_pdv.

**Execution Order:**
1. Ingestion
2. Validation
3. Monitoring

**Alerts:**
- Execution errors are captured and displayed for each step.
- If any step fails, subsequent steps are not executed.

In [ ]:
import sys
sys.path.append("/Workspace/Users/diego.mayorgacapera@gmail.com/.bundle/BI_Market_Visibility/dev/files")
# Import orchestrated functions
from src.bronze.master_pdv.ingest_master_pdv import run_ingestion
from src.bronze.master_pdv.validate_master_pdv import run_validation
from src.bronze.master_pdv.monitor_master_pdv import run_monitoring

In [ ]:
# --- Logging setup (must be first for consistent context)
import logging
logger = logging.getLogger('bronze_orchestrator')
if not logger.hasHandlers():
    logger.setLevel(logging.INFO)
    handler = logging.StreamHandler()
    formatter = logging.Formatter('%(asctime)s %(levelname)s %(name)s: %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)

In [ ]:
# --- Environment Parameters ---
# You can change these for staging/prod
# Bronze table and validation table names must match your deployment
# Example: env = "prod"
# Example: bronze_table = "workspace.bronze.master_pdv"
# Example: validation_table = "workspace.bronze.master_pdv_validation"
env = "dev"
bronze_table = "workspace.bronze.master_pdv"
validation_table = "workspace.bronze.master_pdv_validation"

In [ ]:
# Step 1: Ingestion
try:
    batch_id, rows_ingested = run_ingestion(source_path="/Volumes/workspace/raw_data/master_pdv", delta_table=bronze_table, env=env)
    print(f'✅ Ingestion completed. Batch ID: {batch_id}, Rows: {rows_ingested}')
except Exception as e:
    print(f'❌ Ingestion failed: {str(e)}')
    raise

In [ ]:
# Step 2: Validation (only if ingestion succeeded)
try:
    metrics_df = run_validation(bronze_table=bronze_table, env=env)
    print('✅ Validation completed.')
    display(metrics_df)
except Exception as e:
    print(f'❌ Validation failed: {str(e)}')
    raise

In [ ]:
# Step 3: Monitoring (only if validation succeeded)
try:
    metrics_json, alerts_json = run_monitoring(bronze_table=bronze_table, validation_table=validation_table, env=env)
    print('✅ Monitoring completed.')
    print('Metrics:')
    print(metrics_json)
    print(f'Metrics volume: {len(metrics_json)}')
    if alerts_json:
        print('⚠️ Alerts:')
        print(alerts_json)
        print(f'Alerts volume: {len(alerts_json)}')
    else:
        print('No alerts triggered.')
except Exception as e:
    print(f'❌ Monitoring failed: {str(e)}')
    raise